# MVGFormer on Fit3D: Multi-View Inference and Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nestpr/fit3d-3d-pose-benchmark/blob/main/notebooks/03_mvgformer_inference.ipynb)

This notebook reconstructs 3D human poses from four synchronized and calibrated RGB cameras with the pretrained **MVGFormer** model and evaluates the predictions against the motion-capture ground truth of the **Fit3D** dataset. It is part of the code accompanying the paper *"Comprehensive Evaluation of Monocular and Multi-view 3D Human Pose Estimation Models for Fitness Exercise Analysis"* ([arXiv:2611.02929](https://arxiv.org/abs/2611.02929)).

Unlike the monocular models (TCPFormer, PoseMamba-L), MVGFormer does not use precomputed 2D poses: it takes the images of all cameras and their calibration and estimates the 3D pose of every frame independently.

| | |
|---|---|
| **Model** | MVGFormer, `q1024` configuration (1024 pose queries), pretrained on CMU Panoptic and used without fine-tuning, with mean fusion across the four Fit3D camera views |
| **Input** | RGB videos of the four Fit3D cameras (`50591643`, `58860488`, `60457274`, `65906101`) and their calibration parameters |
| **Output** | 3D predictions of shape `(T, 15, 3)` in millimetres (COMMON15 skeleton, pelvis-centred), and per-sequence and per-joint error tables (MPJPE, PA-MPJPE) |
| **Experiment** | 224 sequences: 8 subjects × 28 exercises, warm-up sequences excluded, every frame processed |
| **Hardware** | GPU runtime (Tesla T4 in the paper). Four views per frame and 1024 queries make this the most expensive model of the study: the complete experiment is long, but the run is resumable. The CUDA operator of the model is compiled at setup |

**Contents:** 1. Requirements · 2. Environment setup · 3. Configuration · 4. Runtime and model loading · 5. Data and joint representation · 6. Multi-view inference · 7. Evaluation metrics · 8. Running the experiment · 9. Output verification · 10. Summary

## 1. Requirements

**Runtime.** A GPU runtime is required, because MVGFormer relies on a custom CUDA operator (deformable projective attention) that is compiled during the setup. The experiments of the paper were run in **Google Colab** (the reference environment) with Python 3.12, PyTorch 2.10, CUDA 12.8 and a Tesla T4. All inputs and outputs are stored under a single root folder, `STORAGE_ROOT`: Google Drive when the notebook runs in Colab, and the folder `data` next to the notebook otherwise.

**Running locally.** The notebook can also be run in Jupyter or VS Code, but only on Linux with an NVIDIA GPU, `git` and the CUDA toolkit (`nvcc`) matching the installed PyTorch, which are needed to compile the operator. The Drive mounting is skipped automatically, MVGFormer is cloned into the folder `external`, and `STORAGE_ROOT` (Section 3) must point to a directory with the layout below. Local execution has not been tested on all systems.

**Data.** Download Fit3D from <https://fit3d.imar.ro> (academic license). The notebook expects:

```text
STORAGE_ROOT/
├── VIDEOS/<subject>/videos/<camera_id>/<exercise>.mp4      # RGB videos of the four cameras
├── camera_params/<subject>/<camera_id>/<exercise>.json     # calibration (intrinsics and extrinsics)
├── Fit3D_2D/<subject>/gt/<exercise>.json                   # Fit3D 3D ground truth (joints3d_25)
└── checkpoints/mvgformer_q1024_model.pth.tar               # pretrained MVGFormer weights
```

The videos are also accepted as `VIDEOS/<subject>/<camera_id>/<exercise>.mp4`, and the 3D ground truth as `VIDEOS/<subject>/<exercise>/joints3d_25.json` (the location used by the monocular notebooks). The checkpoint is also searched in `Fit3D_eval/checkpoints/`, in `STORAGE_ROOT` and in `TFG_MVP/models/MVGFormer/`.

**Checkpoint.** The weights `mvgformer_q1024_model.pth.tar` are released by the MVGFormer authors (<https://github.com/XunshanMan/MVGFormer>). Neither the MVGFormer code nor its weights are redistributed here; they keep their original license. The file `tpose.pt` (canonical T-pose used to initialize the decoder) is downloaded automatically from the same repository.

**How to run.** Execute the cells from top to bottom. Sequences that were already processed are skipped, so an interrupted run can be resumed. For a quick check, set `SUBJECTS`, `MAX_SEQUENCES_PER_SUBJECT` and `MAX_FRAMES_PER_VIDEO` in Section 3.

## 2. Environment Setup

MVGFormer is treated as an external dependency: its code is fetched at run time, whereas the data, the checkpoint and all results live in `STORAGE_ROOT`. The original repository needs a few adaptations to run in a current Colab environment, which are applied by the notebook. None of them changes the pretrained weights; the adaptation of the view fusion to four cameras is described in Section 4.1:

1. **Repository and dependencies.** The repository is cloned into the working directory (`/content` in Colab, `external` locally), its Python dependencies are installed and its internal modules are added to the import path.
2. **Minimal `mmcv`.** The inference pipeline only needs `mkdir_or_exist`, `dump`, `load` and `get_dist_info` from `mmcv`. A minimal replacement module providing them is created, so that the complete `mmcv` package, which is hard to build, is not required.
3. **CUDA operator.** The sources of the `Deformable` operator use PyTorch APIs that were deprecated or removed in recent versions (`Tensor.type()`, `Tensor.data<T>()`, `AT_ASSERTM` and the `THC` atomics header). They are replaced by their current equivalents (`scalar_type()`, `is_cuda()`, `data_ptr<T>()`, `TORCH_CHECK`, `ATen/cuda/Atomic.cuh`), and the operator is compiled only for the architecture of the GPU in use, because the original `setup.py` also targets old architectures that recent CUDA toolkits no longer support. The replacements are semantically identical, so the behaviour of the operator is unchanged. If the compilation fails, the compiler messages are printed. Additionally, an unused `from nis import cat` import in `dq_transformer.py` is removed, because the `nis` module no longer exists in Python 3.13.

After the compilation, the operator is imported and executed on small synthetic tensors (Section 2.1). The notebook should only be continued if this test succeeds.

In [ ]:
# Detect the runtime and prepare the working directory

import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

try:
    import google.colab   # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Directory of the notebook (the working directory when the notebook starts). It is kept if this cell is run
# again, because later cells change the working directory.
PROJECT_DIR = globals().get("PROJECT_DIR") or os.getcwd()

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_DIR = Path("/content")                    # third-party code is cloned here
else:
    WORK_DIR = Path(PROJECT_DIR) / "external"      # third-party code is cloned here
    WORK_DIR.mkdir(parents=True, exist_ok=True)

MVGFORMER_ROOT = WORK_DIR / "MVGFormer"

print(f"Runtime: {'Google Colab' if IN_COLAB else 'local'} | working directory: {WORK_DIR}")

In [ ]:
# Clone MVGFormer, install the dependencies and create the minimal mmcv module

import textwrap


def run_command(command, cwd=None):
    """Run a shell command and raise an error with its output if it fails."""
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        raise RuntimeError(f"Command failed: {' '.join(map(str, command))}")


if not MVGFORMER_ROOT.exists():
    run_command(["git", "clone", "-q", "https://github.com/XunshanMan/MVGFormer.git"], cwd=WORK_DIR)

# Revision of the original code. It is printed so that it can be recorded; set MVGFORMER_COMMIT to a commit hash
# to reproduce exactly that revision (it must be set before the first run, because the repository is cloned once).
MVGFORMER_COMMIT = None

if MVGFORMER_COMMIT:
    run_command(["git", "checkout", "-q", MVGFORMER_COMMIT], cwd=MVGFORMER_ROOT)

commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=MVGFORMER_ROOT, capture_output=True, text=True).stdout.strip()
print("MVGFormer repository commit:", commit)

run_command(
    [sys.executable, "-m", "pip", "install", "-q",
     "json_tricks", "easydict", "prettytable", "pyyaml", "scipy", "matplotlib", "tensorboardX", "h5py", "ninja"]
)

# Internal modules of MVGFormer (model definition, custom operators, validation utilities)
REPO_PATHS = [
    str(MVGFORMER_ROOT / "lib" / "models" / "ops"),
    str(MVGFORMER_ROOT / "lib"),
    str(MVGFORMER_ROOT),
    str(MVGFORMER_ROOT / "run"),
]

for path in REPO_PATHS:
    if path not in sys.path:
        sys.path.insert(0, path)

MMCV_INIT = """
import os
import json
import pickle


def mkdir_or_exist(dir_name):
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)


def _infer_format(file, file_format):
    if file_format is not None:
        return file_format

    ext = os.path.splitext(file)[1].lower()
    return "pkl" if ext in [".pkl", ".pickle"] else "json"


def dump(obj, file, file_format=None):
    file_format = _infer_format(file, file_format)

    if file_format == "json":
        with open(file, "w", encoding="utf-8") as f:
            json.dump(obj, f)
    elif file_format in ["pkl", "pickle"]:
        with open(file, "wb") as f:
            pickle.dump(obj, f)
    else:
        raise ValueError(f"Unsupported format: {file_format}")


def load(file, file_format=None):
    file_format = _infer_format(file, file_format)

    if file_format == "json":
        with open(file, "r", encoding="utf-8") as f:
            return json.load(f)
    elif file_format in ["pkl", "pickle"]:
        with open(file, "rb") as f:
            return pickle.load(f)
    else:
        raise ValueError(f"Unsupported format: {file_format}")
"""

MMCV_RUNNER_INIT = """
def get_dist_info():
    return 0, 1
"""

mmcv_root = MVGFORMER_ROOT / "mmcv"
(mmcv_root / "runner").mkdir(parents=True, exist_ok=True)
(mmcv_root / "__init__.py").write_text(MMCV_INIT, encoding="utf-8")
(mmcv_root / "runner" / "__init__.py").write_text(MMCV_RUNNER_INIT, encoding="utf-8")

print(f"MVGFormer repository: {MVGFORMER_ROOT}")
print(f"Minimal mmcv module:  {mmcv_root}")

In [ ]:
# Patch and compile the Deformable CUDA operator

import re

import torch

OPS_DIR = MVGFORMER_ROOT / "lib" / "models" / "ops"
CU_PATH = OPS_DIR / "src" / "cuda" / "deform_cuda.cu"
CUH_PATH = OPS_DIR / "src" / "cuda" / "deform_im2col_cuda.cuh"
H_PATH = OPS_DIR / "src" / "deform.h"
SETUP_PATH = OPS_DIR / "setup.py"
BUILD_LOG = WORK_DIR / "deformable_build.log"

for path in (CU_PATH, CUH_PATH, SETUP_PATH):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")


def patch_file(path, replacements=(), regexes=()):
    """Apply plain and regular-expression replacements to a text file (idempotent)."""
    text = path.read_text(encoding="utf-8")

    for old, new in replacements:
        text = text.replace(old, new)

    for pattern, new in regexes:
        text = re.sub(pattern, new, text)

    path.write_text(text, encoding="utf-8")


# Deprecated or removed PyTorch APIs: replace them by their current equivalents (same semantics)
#   Tensor.type().is_cuda() -> Tensor.is_cuda()      Tensor.type() (dispatch) -> Tensor.scalar_type()
#   Tensor.data<T>() -> Tensor.data_ptr<T>()          AT_ASSERTM -> TORCH_CHECK
#   <THC/THCAtomics.cuh> (removed from PyTorch)       -> <ATen/cuda/Atomic.cuh>
patch_file(
    CU_PATH,
    replacements=[("value.type()", "value.scalar_type()"), (".data<", ".data_ptr<"), ("AT_ASSERTM(", "TORCH_CHECK(")],
    regexes=[(r"(\w+)\.scalar_type\(\)\.is_cuda\(\)", r"\1.is_cuda()"), (r"(\w+)\.type\(\)\.is_cuda\(\)", r"\1.is_cuda()")],
)
patch_file(CUH_PATH, replacements=[("#include <THC/THCAtomics.cuh>", "#include <ATen/cuda/Atomic.cuh>")])

if H_PATH.exists():
    patch_file(H_PATH, regexes=[(r"(\w+)\.type\(\)\.is_cuda\(\)", r"\1.is_cuda()")])

# The original setup.py compiles for fixed architectures (sm_60 ... sm_75), some of which are not supported by recent
# CUDA toolkits. Compile only for the architecture of the GPU in use.
if "get_device_capability" not in SETUP_PATH.read_text(encoding="utf-8"):
    patch_file(
        SETUP_PATH,
        regexes=[(
            r'(?s)extra_compile_args\["nvcc"\] = \[.*?\n        \]',
            'capability = "".join(str(v) for v in torch.cuda.get_device_capability())\n'
            '        extra_compile_args["nvcc"] = [\n'
            '            "-DCUDA_HAS_FP16=1",\n'
            '            "-D__CUDA_NO_HALF_OPERATORS__",\n'
            '            "-D__CUDA_NO_HALF_CONVERSIONS__",\n'
            '            "-D__CUDA_NO_HALF2_OPERATORS__",\n'
            '            f"-gencode=arch=compute_{capability},code=sm_{capability}",\n'
            '        ]',
        )],
    )

if "get_device_capability" not in SETUP_PATH.read_text(encoding="utf-8"):
    raise RuntimeError("Could not patch the GPU architecture flags in setup.py (unexpected file layout).")

# `from nis import cat` is an unused import (the code calls torch.cat); the `nis` module was removed in Python 3.13
DQ_PATH = MVGFORMER_ROOT / "lib" / "models" / "dq_transformer.py"
patch_file(DQ_PATH, replacements=[("from nis import cat\n", "")])

# Start from a clean build
shutil.rmtree(OPS_DIR / "build", ignore_errors=True)

for stale in OPS_DIR.glob("*.so"):
    stale.unlink()

nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print("PyTorch", torch.__version__, "| CUDA (PyTorch):", torch.version.cuda, "| GPU capability:", torch.cuda.get_device_capability() if torch.cuda.is_available() else "none")
print("nvcc:", nvcc.stdout.strip().splitlines()[-1] if nvcc.returncode == 0 else "not found")

with open(BUILD_LOG, "w", encoding="utf-8") as log:
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=OPS_DIR, stdout=log, stderr=subprocess.STDOUT,
    )

so_files = sorted(OPS_DIR.glob("Deformable*.so"))

if len(so_files) == 0:
    lines = BUILD_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    error_lines = [line for line in lines if "error" in line.lower() and "Traceback" not in line]
    print("Compilation failed. Compiler messages containing 'error' (first 40):")
    print("\n".join(error_lines[:40]))
    print("\nLast lines of the build log:")
    print("\n".join(lines[-15:]))
    raise RuntimeError(f"Deformable*.so was not generated. Full log: {BUILD_LOG}")

print("Deformable compilation completed.")
print("Generated extension:", so_files[0])
print("Build log:", BUILD_LOG)

### 2.1 Deformable CUDA Validation

The compiled operator is validated before the model is loaded. The cell

- adds the PyTorch shared-library directory to `LD_LIBRARY_PATH` and preloads the PyTorch and CUDA libraries required by the extension,
- removes previously imported versions of the deformable-attention modules from the interpreter,
- imports the new `Deformable` extension and checks that `deform_forward` is available,
- reloads the Python wrapper of MVGFormer (`deform_func`) and imports the `ProjAttn` module,
- executes `deform_forward` on small synthetic tensors.

If any of these steps fails, the notebook stops with an error.

In [ ]:
# Validate the compiled Deformable CUDA operator

import ctypes
import glob
import importlib

import torch

TORCH_LIB = Path(torch.__file__).parent / "lib"
os.environ["LD_LIBRARY_PATH"] = str(TORCH_LIB) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

for libname in ["libc10.so", "libtorch.so", "libtorch_cpu.so", "libtorch_python.so", "libc10_cuda.so", "libtorch_cuda.so"]:
    libpath = TORCH_LIB / libname

    if libpath.exists():
        ctypes.CDLL(str(libpath), mode=ctypes.RTLD_GLOBAL)

for module_name in list(sys.modules.keys()):
    if module_name == "Deformable" or module_name.startswith(("models.ops", "lib.models.ops")):
        del sys.modules[module_name]

importlib.invalidate_caches()

if len(glob.glob(str(OPS_DIR / "Deformable*.so"))) == 0:
    raise FileNotFoundError("Deformable*.so was not found. Run the compilation cell first.")

import Deformable

if not hasattr(Deformable, "deform_forward"):
    raise RuntimeError("The Deformable extension was imported but does not provide deform_forward.")

from models.ops.functions import deform_func
importlib.reload(deform_func)

HAS_DEFORMABLE_EXT = True
deform_func.HAS_DEFORMABLE_EXT = HAS_DEFORMABLE_EXT

from models.ops.modules import ProjAttn

# Forward pass on synthetic tensors
test_device = "cuda" if torch.cuda.is_available() else "cpu"
N, H, W, M, D, Lq, L, P = 1, 2, 2, 1, 4, 3, 1, 2

value = torch.randn(N, H * W, M, D, device=test_device, dtype=torch.float32)
spatial_shapes = torch.tensor([[H, W]], device=test_device, dtype=torch.long)
level_start_index = torch.tensor([0], device=test_device, dtype=torch.long)
sampling_locations = torch.rand(N, Lq, M, L, P, 2, device=test_device, dtype=torch.float32)
attention_weights = torch.rand(N, Lq, M, L, P, device=test_device, dtype=torch.float32)
attention_weights = attention_weights / attention_weights.sum(dim=(-1, -2), keepdim=True)

output = Deformable.deform_forward(value, spatial_shapes, level_start_index, sampling_locations, attention_weights, 64)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("Deformable CUDA available:", HAS_DEFORMABLE_EXT)
print("Loaded extension:", Deformable.__file__)
print("deform_forward test:", tuple(output.shape), output.device, output.dtype)

In [ ]:
# Imports

import contextlib
import gc
import glob
import io
import json
import platform
import random
import time
import traceback

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 3. Configuration

All user-editable settings are defined in the next cell; the rest of the notebook only reads them.

| Setting | Meaning |
|---|---|
| `STORAGE_ROOT` | Root folder of all inputs and outputs: Google Drive in Colab, the folder `data` otherwise. Edit it for a local run. |
| `EXCLUDE_WARMUPS` | Exclude the warm-up sequences, as in the paper. |
| `SUBJECTS`, `EXERCISES`, `MAX_SEQUENCES_PER_SUBJECT` | Optional subset of the sequences. With the default `None` the complete experiment (224 sequences) is run. |
| `FRAME_STRIDE`, `MAX_FRAMES_PER_VIDEO` | Frame sampling. The paper uses every frame (`1`) of the complete sequence (`None`). Other values are intended for quick tests. |
| `BATCH_CANDIDATES` | Numbers of frames processed simultaneously. The first value is used; if the GPU runs out of memory the sequence is repeated with the next one. It only affects speed and memory. |
| `USE_GT_IN_META_FOR_COMPAT` | If `True`, the compatibility fields of the view metadata are filled with the ground truth; with the default `False` they are filled with zeros and **no ground truth is given to the model** (see Section 6.3). The predictions are the same in both cases. |
| `SEED` | Random seed. It is reset before every sequence, so the result of a sequence does not depend on the order of the run or on skipped sequences. |
| `USE_LOCAL_VIDEO_CACHE` | Copy the four videos of a sequence to the local disk before reading them, which is much faster than reading from Google Drive. The copies are deleted after each sequence when `CLEAR_VIDEO_CACHE` is enabled. |
| `STOP_ON_FIRST_ERROR` | Stop the run at the first sequence that fails. By default the error is reported and the run continues. |

The model runs in FP32 (no mixed precision). The remaining lines of the cell derive the input and output locations. The pretrained checkpoint is searched in several standard locations and the first one found is used.

In [ ]:
# ----------------------------------------------------------------------------
# User-editable settings
# ----------------------------------------------------------------------------
STORAGE_ROOT = "/content/drive/MyDrive" if IN_COLAB else os.path.join(PROJECT_DIR, "data")   # root folder of all inputs and outputs

EXCLUDE_WARMUPS = True                    # exclude warm-up sequences
SUBJECTS = None                           # e.g. ["s03"] for a quick test; None = all subjects
EXERCISES = None                          # e.g. ["band_pull_apart"]; None = all exercises
MAX_SEQUENCES_PER_SUBJECT = None          # e.g. 3 for a quick test; None = all sequences

FRAME_STRIDE = 1                          # 1 = every frame (paper)
MAX_FRAMES_PER_VIDEO = None               # e.g. 100 for a quick test; None = complete sequence (paper)

BATCH_CANDIDATES = [8, 4, 2, 1]           # frames per forward pass; the next value is tried after an out-of-memory error
USE_GT_IN_META_FOR_COMPAT = False         # False: the metadata fields that exist only for compatibility are zeros (no ground truth enters the model)
SEED = 42

USE_LOCAL_VIDEO_CACHE = True              # copy the videos of a sequence to the local disk before reading them
CLEAR_VIDEO_CACHE = True                  # delete the local copies after each sequence
STOP_ON_FIRST_ERROR = False

# ----------------------------------------------------------------------------
# Derived locations
# ----------------------------------------------------------------------------
STORAGE_ROOT = Path(STORAGE_ROOT)

VIDEO_ROOT = STORAGE_ROOT / "VIDEOS"                  # RGB videos (and 3D ground truth as a fallback)
CAMERA_PARAMS_ROOT = STORAGE_ROOT / "camera_params"   # camera calibration
FIT3D_2D_ROOT = STORAGE_ROOT / "Fit3D_2D"             # 3D ground truth

IS_SUBSET = (
    SUBJECTS is not None or EXERCISES is not None or MAX_SEQUENCES_PER_SUBJECT is not None
    or FRAME_STRIDE != 1 or MAX_FRAMES_PER_VIDEO is not None
)

SAVE_ROOT = STORAGE_ROOT / ("Fit3D_MVGFormer_subset" if IS_SUBSET else "Fit3D_MVGFormer")
PRED_DIR = SAVE_ROOT / "predictions_3d"
CSV_DIR = SAVE_ROOT / "csv"
PER_VIDEO_CSV = CSV_DIR / "results_per_video_mvgformer_video_direct.csv"
PER_JOINT_CSV = CSV_DIR / "results_per_joint_mvgformer_video_direct.csv"

POSE_SOURCE_NAME = "multiview_rgb"
# The tag identifies the configuration of the run. It includes how the metadata fields were filled, so results obtained
# with a different setting are never reused when a run is resumed.
RUN_TAG = f"video_direct_4cam_stride{FRAME_STRIDE}" + ("_gtmeta" if USE_GT_IN_META_FOR_COMPAT else "_nogt")

LOCAL_VIDEO_CACHE_ROOT = WORK_DIR / "tmp_mvgformer_videos"

CHECKPOINT_NAME = "mvgformer_q1024_model.pth.tar"
CHECKPOINT_CANDIDATES = [
    STORAGE_ROOT / "checkpoints" / CHECKPOINT_NAME,
    STORAGE_ROOT / "Fit3D_eval" / "checkpoints" / CHECKPOINT_NAME,
    STORAGE_ROOT / CHECKPOINT_NAME,
    STORAGE_ROOT / "TFG_MVP" / "models" / "MVGFormer" / CHECKPOINT_NAME,
]
CHECKPOINT_PATH = next((path for path in CHECKPOINT_CANDIDATES if path.is_file()), CHECKPOINT_CANDIDATES[0])
TPOSE_PATH = MVGFORMER_ROOT / "tpose.pt"

CAMERA_IDS = ["50591643", "58860488", "60457274", "65906101"]
CAM_ID_TO_CAM_NAME = {"50591643": "cam0", "58860488": "cam1", "60457274": "cam2", "65906101": "cam3"}

assert FRAME_STRIDE >= 1, "FRAME_STRIDE must be a positive integer."
assert len(BATCH_CANDIDATES) > 0 and all(size >= 1 for size in BATCH_CANDIDATES), "BATCH_CANDIDATES must contain positive integers."

for folder in (PRED_DIR, CSV_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected; MVGFormer needs a GPU runtime.")

print(f"Videos:      {VIDEO_ROOT}")
print(f"Calibration: {CAMERA_PARAMS_ROOT}")
print(f"Checkpoint:  {CHECKPOINT_PATH}")
print(f"Outputs:     {SAVE_ROOT}" + ("   (subset run)" if IS_SUBSET else ""))
print(f"Frames: stride {FRAME_STRIDE}, maximum per video {MAX_FRAMES_PER_VIDEO} | batch sizes: {BATCH_CANDIDATES}")

## 4. Runtime and Model Loading

### 4.1 Model configuration

MVGFormer is configured for the Fit3D multi-view setup by means of a YAML file that is loaded with the configuration system of the original repository and then completed with the final values used in the experiments:

- **4 synchronized camera views** and **15 output joints**.
- Input images of **960 × 540** pixels and heatmaps of **240 × 135**.
- **1024 person queries** (matching the `q1024` checkpoint), **4 decoder layers**, feature levels `[0, 1, 2]`, 8 deformable-attention sampling points and mean fusion across views. The configuration file of the official repository (`knn5-lr4-q1024.yaml`) is defined for 5 cameras (CMU Panoptic) and sets `fuse_view_feats: 'cat_proj'`, a learned projection of the concatenated view features whose size depends on the number of cameras. Fit3D has 4 cameras, so this notebook uses `'mean'`, which does not depend on the number of views. This is a **configuration choice for four cameras**, and it does not discard any pretrained weight: when the released `q1024` checkpoint is loaded (Section 4.2), all of the 536 model tensors are loaded and **no checkpoint parameter is ignored** (verified on a Tesla T4), i.e. the checkpoint contains no view-fusion projection. Any checkpoint parameter that the model does not use would be listed when the model is loaded.
- Data augmentation disabled.
- A 3D search volume centred at $[-25.3935,\ 170.7180,\ 990.6322]$ mm with dimensions $[2200,\ 1600,\ 2400]$ mm, which contains the capture area of Fit3D.

The random seed is applied to Python, NumPy and PyTorch.

In [ ]:
# Prepare the MVGFormer runtime

os.chdir(MVGFORMER_ROOT)

from models.ops.functions.deform_func import HAS_DEFORMABLE_EXT
from models.ops.modules import ProjAttn
from models import multi_view_pose_transformer
from run import validate_3d

print("HAS_DEFORMABLE_EXT =", HAS_DEFORMABLE_EXT)
print("ProjAttn: OK")
print("multi_view_pose_transformer: OK")
print("validate_3d: OK")

In [ ]:
# Create the Fit3D-specific MVGFormer configuration file

CONFIG_DIR = MVGFORMER_ROOT / "configs" / "fit3d"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_TEXT = """
CUDNN:
  BENCHMARK: true
  DETERMINISTIC: false
  ENABLED: true

WORKERS: 2
PRINT_FREQ: 20

DATASET:
  TRAIN_DATASET: 'fit3d'
  TEST_DATASET: 'fit3d'
  TRAIN_SUBSET: 'train'
  TEST_SUBSET: 'test'
  ROOT: '__DATASET_ROOT__'
  CAMERA_NUM: 4
  ROOTIDX: 2
  DATA_FORMAT: 'jpg'
  DATA_AUGMENTATION: false
  SCALE_FACTOR: 0.0
  ROT_FACTOR: 0.0
  FLIP: false
  COLOR_RGB: true

NETWORK:
  NUM_JOINTS: 15
  IMAGE_SIZE: [960, 540]
  HEATMAP_SIZE: [240, 135]
  SIGMA: 2
  TARGET_TYPE: 'gaussian'

LOSS:
  USE_DIFFERENT_JOINTS_WEIGHT: false

MULTI_PERSON:
  MAX_PEOPLE_NUM: 10
  SPACE_SIZE: [4000, 4000, 4000]
  SPACE_CENTER: [0, 0, 0]
  INITIAL_CUBE_SIZE: [20, 20, 20]

MODEL: 'multi_person_posenet'

DECODER:
  d_model: 256
  dim_feedforward: 1024
  dropout: 0.1
  activation: 'relu'
  num_feature_levels: 1
  use_feat_level: [0, 1, 2]
  nhead: 8
  dec_n_points: 8
  detach_refpoints_cameraprj_firstlayer: true
  fuse_view_feats: 'mean'
  num_decoder_layers: 4
  return_intermediate_dec: true
  num_instance: 1024
  num_keypoints: 15
  query_embed_type: 'person_joint'
  projattn_posembed_mode: 'ablation_not_use_rayconv'

TEST:
  BATCH_SIZE: 1
"""

# The dataset root is not read during the video-based inference, but the configuration requires an existing entry
CONFIG_PATH = CONFIG_DIR / "mvgformer_fit3d.yaml"
CONFIG_PATH.write_text(CONFIG_TEXT.replace("__DATASET_ROOT__", str(VIDEO_ROOT)), encoding="utf-8")

print("Configuration created:", CONFIG_PATH)

In [ ]:
# Load and finalize the MVGFormer configuration

from core.config import config, update_config

update_config(str(CONFIG_PATH))

if not hasattr(config.DATASET, "FIT3D_EXERCISES"):
    config.DATASET.FIT3D_EXERCISES = []

config.DATASET.CAMERA_NUM = 4
config.DATASET.ROOTIDX = 2
config.DATASET.DATA_FORMAT = "jpg"
config.DATASET.DATA_AUGMENTATION = False
config.DATASET.SCALE_FACTOR = 0.0
config.DATASET.ROT_FACTOR = 0.0
config.DATASET.FLIP = False
config.DATASET.COLOR_RGB = True

config.NETWORK.NUM_JOINTS = 15
config.NETWORK.IMAGE_SIZE = [960, 540]
config.NETWORK.HEATMAP_SIZE = [240, 135]

config.DECODER.num_keypoints = 15
config.DECODER.num_instance = 1024
config.DECODER.num_decoder_layers = 4
config.DECODER.use_feat_level = [0, 1, 2]
config.DECODER.dec_n_points = 8
config.DECODER.projattn_posembed_mode = "ablation_not_use_rayconv"
config.DECODER.convert_joint_format_indices = None

config.DEBUG.LOG_VAL_LOSS = False
config.DEBUG.VISUALIZATION_JUMP_NUM = -1

config.MULTI_PERSON.MAX_PEOPLE_NUM = 10
config.MULTI_PERSON.SPACE_CENTER = [-25.3935, 170.7180, 990.6322]
config.MULTI_PERSON.SPACE_SIZE = [2200.0, 1600.0, 2400.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed):
    """Set the random seed of Python, NumPy and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

print("MVGFormer configuration loaded:", CONFIG_PATH)
print("Device:", device)
print("NUM_JOINTS:", config.NETWORK.NUM_JOINTS, "| IMAGE_SIZE:", config.NETWORK.IMAGE_SIZE)
print("num_instance:", config.DECODER.num_instance, "| use_feat_level:", config.DECODER.use_feat_level)
print("SPACE_CENTER:", config.MULTI_PERSON.SPACE_CENTER, "| SPACE_SIZE:", config.MULTI_PERSON.SPACE_SIZE)

### 4.2 Pretrained model

The experiment uses the pretrained **MVGFormer q1024** checkpoint together with the T-pose initialization of the decoder. The T-pose file is downloaded from the MVGFormer repository when it is not available, whereas the checkpoint must be present in `STORAGE_ROOT` (Section 1).

The model is instantiated with the configuration above and the checkpoint is loaded as follows:

- A possible `module.` prefix introduced by multi-GPU training is removed from the parameter names.
- Only parameters whose names exist in the model are considered, and those with incompatible shapes are skipped (and reported).
- The compatible weights are loaded with `strict=False`, and the `instance_embedding` weights are copied explicitly.
- The reference-point initialization is set to `sample_space`, as in the Fit3D experiments.

After loading, the network is switched to evaluation mode. The cell reports the number of loaded, skipped (shape mismatch), missing and ignored parameters. A correct load has **no skipped and no missing parameters**; otherwise the notebook stops (`STRICT_CHECKPOINT_LOADING`). *Ignored* parameters are stored in the checkpoint but do not exist in the model (for example, a view-fusion projection, see Section 4.1). With the released `q1024` checkpoint there are none (536/536 model tensors loaded, 0 ignored); the list is printed so that this can be checked.

In [ ]:
# Load the pretrained MVGFormer q1024 model

from collections import OrderedDict
from models import dq_transformer as dqt

if not TPOSE_PATH.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/XunshanMan/MVGFormer/master/tpose.pt", str(TPOSE_PATH))

if not TPOSE_PATH.exists():
    raise FileNotFoundError(f"File not found: {TPOSE_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"MVGFormer checkpoint not found: {CHECKPOINT_PATH}\n"
        f"Place '{CHECKPOINT_NAME}' in {STORAGE_ROOT / 'checkpoints'} (see Section 1)."
    )

config.DECODER.t_pose_dir = str(TPOSE_PATH)
config.TEST.MODEL_FILE = str(CHECKPOINT_PATH)
config.DECODER.num_instance = 1024

if hasattr(config.DECODER, "NUM_INSTANCE"):
    config.DECODER.NUM_INSTANCE = 1024

if hasattr(config.NETWORK, "NUM_QUERY"):
    config.NETWORK.NUM_QUERY = 1024

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = dqt.get_mvp(config, is_train=False).to(device)

checkpoint = torch.load(str(CHECKPOINT_PATH), map_location=device, weights_only=False)
state = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint

state_clean = OrderedDict(
    (key[7:] if key.startswith("module.") else key, value)
    for key, value in state.items()
)

STRICT_CHECKPOINT_LOADING = True   # stop if a model parameter is missing or has an incompatible shape

model_state = model.state_dict()
loadable = OrderedDict()
skipped = []
ignored_keys = [key for key in state_clean if key not in model_state]   # in the checkpoint, not in the adapted model

for key, value in state_clean.items():
    if key not in model_state:
        continue

    if tuple(model_state[key].shape) != tuple(value.shape):
        skipped.append((key, tuple(value.shape), tuple(model_state[key].shape)))
        continue

    loadable[key] = value

missing_keys, unexpected_keys = model.load_state_dict(loadable, strict=False)

if "instance_embedding.weight" in state_clean:
    with torch.no_grad():
        checkpoint_instances = state_clean["instance_embedding.weight"].to(device)
        n_instances = min(model.instance_embedding.weight.shape[0], checkpoint_instances.shape[0])
        model.instance_embedding.weight[:n_instances].copy_(checkpoint_instances[:n_instances])
else:
    n_instances = 0

if hasattr(model, "init_ref_method"):
    model.init_ref_method = "sample_space"
    model.init_ref_method_value = None

# Feature levels used by the experiments
config.DECODER.use_feat_level = [0, 1, 2]

if hasattr(model, "use_feat_level"):
    model.use_feat_level = [0, 1, 2]

model.eval()
device = next(model.parameters()).device

print("MVGFormer model loaded.")
print("Device:", device)
print("Checkpoint:", CHECKPOINT_PATH)
print("T-pose:", TPOSE_PATH)
print("Number of joints:", model.num_joints)
print("Instance embedding:", tuple(model.instance_embedding.weight.shape))
print("Loaded parameters:", len(loadable))
print("Skipped parameters (shape mismatch):", len(skipped))
print("Missing parameters:", len(missing_keys))
print("Checkpoint parameters not used by this model:", len(ignored_keys))

for key in ignored_keys[:20]:
    print("   ", key)

if len(ignored_keys) > 20:
    print(f"    ... and {len(ignored_keys) - 20} more")

if STRICT_CHECKPOINT_LOADING and (len(skipped) > 0 or len(missing_keys) > 0):
    raise RuntimeError(
        f"Incomplete checkpoint load: {len(skipped)} parameters with an incompatible shape and "
        f"{len(missing_keys)} model parameters missing in the checkpoint. Inference has been stopped."
    )
print("Copied instance embeddings:", n_instances)

## 5. Data and Joint Representation

### 5.1 Common 15-joint skeleton

MVGFormer and Fit3D do not use the same joint set or order. MVGFormer predicts 15 joints in the Panoptic-style order, whereas the Fit3D ground truth (`joints3d_25`) provides the 17-joint layout shared with the monocular models. Both are converted to a common 15-joint skeleton, **COMMON15**, before any error is computed:

`pelvis, rhip, rknee, rankle, lhip, lknee, lankle, neck, nose, lsho, lelb, lwri, rsho, relb, rwri`

- **Ground truth.** The joints of the 17-joint layout with indices `FIT3D17_TO_COMMON15` are selected. The spine (index 7) and the head (index 10) are discarded because MVGFormer has no equivalent joints, whereas the nose (index 9) is retained, as in the paper.
- **Prediction.** The 15 raw joints of MVGFormer are reordered with `MVG_RAW_TO_COMMON15`.
- **Units.** Coordinates are converted from metres to millimetres when their magnitude indicates metres (`maybe_to_mm`).
- **Arm convention.** The left and right arms of the Fit3D ground truth are exchanged before the evaluation, to match the convention of the MVGFormer prediction.
- **Root alignment.** Prediction and ground truth are expressed relative to the pelvis.

The metrics are therefore computed on 15 joints, which is why the PA-MPJPE values of MVGFormer are reported on COMMON15 in the paper and are not directly comparable with the 17-joint values of the monocular notebooks.

In [ ]:
# Joint layouts and conversion utilities

COMMON15_NAMES = [
    "pelvis", "rhip", "rknee", "rankle",
    "lhip", "lknee", "lankle",
    "neck", "nose",
    "lsho", "lelb", "lwri",
    "rsho", "relb", "rwri",
]

# Indices of the 17-joint Fit3D layout that form COMMON15 (the spine and the head are discarded)
FIT3D17_TO_COMMON15 = [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16]

# For each COMMON15 joint, the index of the corresponding raw MVGFormer output joint
MVG_RAW_TO_COMMON15 = [2, 6, 7, 8, 12, 13, 14, 0, 1, 9, 10, 11, 3, 4, 5]

ARM_SWAP_COMMON15 = [(9, 12), (10, 13), (11, 14)]

BONES_COMMON15 = [
    (0, 1), (1, 2), (2, 3),
    (0, 4), (4, 5), (5, 6),
    (0, 7), (7, 8),
    (7, 9), (9, 10), (10, 11),
    (7, 12), (12, 13), (13, 14),
]


def to_numpy(x):
    """Convert a PyTorch tensor or array-like object to NumPy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()

    return np.asarray(x)


def fix_pose(pose):
    """Convert a pose to a consistent (joints, 3) float32 array."""
    pose = to_numpy(pose)

    if pose.ndim == 3 and pose.shape[0] == 1:
        pose = pose[0]

    if pose.ndim == 2 and pose.shape[-1] != 3 and pose.shape[0] == 3:
        pose = pose.T

    return pose.astype(np.float32)


def maybe_to_mm(pose):
    """Convert pose coordinates to millimetres when they appear to be in metres."""
    pose = fix_pose(pose).copy()

    if np.nanmax(np.abs(pose)) < 20:
        pose *= 1000.0

    return pose.astype(np.float32)


def root_align_common15(pose):
    """Root-align a COMMON15 pose using the pelvis joint."""
    pose = fix_pose(pose)
    return (pose - pose[0:1]).astype(np.float32)


def swap_arms_common15(pose):
    """Exchange the left and right arm joints of a COMMON15 pose."""
    pose = fix_pose(pose).copy()

    for left, right in ARM_SWAP_COMMON15:
        pose[[left, right]] = pose[[right, left]]

    return pose.astype(np.float32)


def gt25_to_common15_mm(gt25_frame):
    """Convert one Fit3D joints3d_25 frame to COMMON15 in millimetres."""
    gt25_frame = np.asarray(gt25_frame, dtype=np.float32)

    if gt25_frame.ndim != 2 or gt25_frame.shape[0] < 17 or gt25_frame.shape[1] != 3:
        raise ValueError(f"Unexpected ground-truth shape: {gt25_frame.shape}")

    return maybe_to_mm(gt25_frame[FIT3D17_TO_COMMON15].copy())


def common15_to_mvg_raw(common15):
    """Convert COMMON15 coordinates to the internal MVGFormer joint order."""
    common15 = maybe_to_mm(common15)
    raw = np.zeros((15, 3), dtype=np.float32)

    for common_idx, raw_idx in enumerate(MVG_RAW_TO_COMMON15):
        raw[raw_idx] = common15[common_idx]

    return raw.astype(np.float32)

### 5.2 Locating videos, calibration and ground truth

For every subject and exercise, the notebook locates

1. the RGB video of each of the four cameras,
2. the calibration file of each camera (one JSON file per camera and exercise, with `extrinsics` and `intrinsics_w_distortion`),
3. the Fit3D 3D ground truth (`joints3d_25`).

The standard locations listed in Section 1 are tried first, followed by recursive searches that tolerate the small differences between the prepared Fit3D folders. If the calibration file of an exercise is missing, the file of the same camera for another exercise is used and a warning is printed (the cameras are fixed during a recording session); a file of a different camera is never used.

**Camera parameters.** The extrinsic translation is converted to millimetres when it is expressed in metres, and the intrinsic matrix is built from the focal length and the principal point; the distortion coefficients are kept in the camera metadata. The camera names `cam0`–`cam3` follow the order of `CAMERA_IDS`.

In [ ]:
# Locate Fit3D videos, camera calibrations and ground-truth files

def find_video_file(subject, exercise, cam_id):
    """Locate the RGB video for a subject, exercise and camera."""
    patterns = [
        VIDEO_ROOT / subject / "videos" / cam_id / f"{exercise}.mp4",
        VIDEO_ROOT / subject / cam_id / f"{exercise}.mp4",
        VIDEO_ROOT / subject / "videos" / cam_id / f"*{exercise}*.mp4",
        VIDEO_ROOT / subject / cam_id / f"*{exercise}*.mp4",
        VIDEO_ROOT / f"**/{subject}/**/{cam_id}/**/{exercise}.mp4",
        VIDEO_ROOT / f"**/{subject}/**/{cam_id}/**/*{exercise}*.mp4",
        VIDEO_ROOT / f"**/*{subject}*/*{cam_id}*/*{exercise}*.mp4",
    ]

    for pattern in patterns:
        matches = sorted(glob.glob(str(pattern), recursive=True))

        if matches:
            return Path(matches[0])

    raise FileNotFoundError(f"Video not found for {subject}/{cam_id}/{exercise}")


_CALIBRATION_WARNINGS = set()


def find_camera_param_file(subject, exercise, cam_id):
    """Locate the calibration file for a subject, exercise and camera."""
    candidates = [
        CAMERA_PARAMS_ROOT / subject / cam_id / f"{exercise}.json",
        CAMERA_PARAMS_ROOT / subject / cam_id / f"{exercise} (1).json",
        CAMERA_PARAMS_ROOT / subject / "videos" / cam_id / f"{exercise}.json",
        CAMERA_PARAMS_ROOT / subject / "videos" / cam_id / f"{exercise} (1).json",
    ]

    for path in candidates:
        if path.exists():
            return path

    subject_root = CAMERA_PARAMS_ROOT / subject

    if subject_root.exists():
        # The file must belong to the requested camera; a file of the same exercise is preferred
        scored = []

        for path in subject_root.rglob("*.json"):
            text = str(path).lower()

            if cam_id.lower() in text:
                scored.append((10 + (10 if exercise.lower() in text else 0), path))

        if scored:
            score, chosen = sorted(scored, key=lambda item: (-item[0], str(item[1])))[0]

            if score < 20 and (subject, cam_id, exercise) not in _CALIBRATION_WARNINGS:
                _CALIBRATION_WARNINGS.add((subject, cam_id, exercise))
                print(f"WARNING | {subject}/{exercise} | calibration of camera {cam_id} not found for this exercise; using {chosen}")

            return chosen

    raise FileNotFoundError(f"Calibration not found for {subject}/{cam_id}/{exercise}")


def find_gt_json(subject, exercise):
    """Locate the Fit3D 3D ground-truth JSON file."""
    candidates = [
        FIT3D_2D_ROOT / subject / "gt" / f"{exercise}.json",
        FIT3D_2D_ROOT / subject / f"{exercise}.json",
        VIDEO_ROOT / subject / exercise / "joints3d_25.json",
    ]

    for path in candidates:
        if path.exists():
            return path

    subject_root = FIT3D_2D_ROOT / subject

    if subject_root.exists():
        # The 2D pose folders also contain <exercise>.json files, which are not ground truth
        matches = [
            path for path in sorted(subject_root.rglob(f"{exercise}.json"))
            if not path.parent.name.lower().startswith("pose")
        ]

        if matches:
            return matches[0]

    raise FileNotFoundError(f"Ground-truth JSON not found for {subject}/{exercise}")


def count_frames(video_path):
    """Return the number of frames of a video."""
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Unable to open video: {video_path}")

    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    return n_frames


def normalize_camera_json(json_path):
    """Load a Fit3D calibration file and return the camera parameters in the format of MVGFormer."""
    with open(json_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if "extrinsics" not in data:
        raise KeyError(f"Missing extrinsics in {json_path}")

    R = np.asarray(data["extrinsics"]["R"], dtype=np.float32).reshape(3, 3)
    T = np.asarray(data["extrinsics"]["T"], dtype=np.float32).reshape(3, 1)

    if np.nanmax(np.abs(T)) < 100:
        T *= 1000.0

    if "intrinsics_w_distortion" in data:
        intrinsics = data["intrinsics_w_distortion"]
    elif "intrinsics_wo_distortion" in data:
        intrinsics = data["intrinsics_wo_distortion"]
    else:
        raise KeyError(f"Missing camera intrinsics in {json_path}")

    fxy = np.asarray(intrinsics["f"], dtype=np.float32).reshape(-1)
    cxy = np.asarray(intrinsics["c"], dtype=np.float32).reshape(-1)
    fx, fy = np.float32(fxy[0]), np.float32(fxy[1])
    cx, cy = np.float32(cxy[0]), np.float32(cxy[1])

    k = np.asarray(intrinsics.get("k", [0.0, 0.0, 0.0]), dtype=np.float32).reshape(-1)
    p = np.asarray(intrinsics.get("p", [0.0, 0.0]), dtype=np.float32).reshape(-1)

    if k.size < 3:
        k = np.pad(k, (0, 3 - k.size))

    if p.size < 2:
        p = np.pad(p, (0, 2 - p.size))

    K = np.array([[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]], dtype=np.float32)

    return {
        "R": R.astype(np.float32),
        "T": T.astype(np.float32),
        "fx": fx,
        "fy": fy,
        "cx": cx,
        "cy": cy,
        "K": K,
        "k": k[:3].reshape(3, 1).astype(np.float32),
        "p": p[:2].reshape(2, 1).astype(np.float32),
        "standard_T": (-R @ T).astype(np.float32),
    }


def load_cameras(subject, exercise):
    """Load the calibration of the four cameras, indexed by camera name and by camera identifier."""
    cameras = {}

    for cam_id in CAMERA_IDS:
        camera = normalize_camera_json(find_camera_param_file(subject, exercise, cam_id))
        cameras[CAM_ID_TO_CAM_NAME[cam_id]] = camera
        cameras[cam_id] = camera

    return cameras


def load_ground_truth(subject, exercise):
    """Load the Fit3D ground truth of a sequence as COMMON15 (millimetres) and return it with its path."""
    gt_path = find_gt_json(subject, exercise)

    with open(gt_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if "joints3d_25" not in data:
        raise KeyError(f"joints3d_25 not found in {gt_path}. Keys: {list(data.keys())}")

    gt25 = np.asarray(data["joints3d_25"], dtype=np.float32)

    if gt25.ndim == 4:
        gt25 = gt25[:, 0]

    if gt25.ndim != 3 or gt25.shape[-1] != 3:
        raise ValueError(f"Unexpected joints3d_25 shape: {gt25.shape}")

    gt_common15 = np.stack([gt25_to_common15_mm(frame) for frame in gt25], axis=0)

    return gt_common15.astype(np.float32), gt_path


def to_evaluation_ground_truth(gt_common15):
    """Apply the arm exchange and the pelvis alignment used to evaluate the predictions."""
    return np.stack(
        [root_align_common15(swap_arms_common15(pose)) for pose in gt_common15],
        axis=0,
    ).astype(np.float32)


def select_frame_indices(num_available):
    """Frames evaluated in a sequence, given the number of frames available in all cameras and in the ground truth."""
    indices = list(range(0, int(num_available), FRAME_STRIDE))

    if MAX_FRAMES_PER_VIDEO is not None:
        indices = indices[:MAX_FRAMES_PER_VIDEO]

    return indices

## 6. Multi-View Inference

### 6.1 RGB frame preprocessing

Each frame is transformed to the representation expected by MVGFormer:

1. An affine transformation maps the original frame to the configured input size of **960 × 540** pixels. It preserves the aspect ratio: for the square $900 \times 900$ Fit3D frames the image is scaled uniformly and centred in the input canvas, which introduces lateral padding instead of an anisotropic resize.
2. The frame is warped with bilinear interpolation and converted from BGR to RGB.
3. Pixel values are scaled to $[0, 1]$ and normalized with the ImageNet mean $(0.485, 0.456, 0.406)$ and standard deviation $(0.229, 0.224, 0.225)$.
4. The result is converted to a tensor of shape $(3, 540, 960)$.

The forward and inverse affine transformations are kept because they are part of the view metadata required by the model. The original intrinsic and extrinsic camera parameters are not modified.

In [ ]:
# Preprocess the RGB frames

def get_affine_transform(center, scale, rot, output_size, shift=np.array([0, 0], dtype=np.float32), inv=0):
    """Compute the affine transformation that maps the input frame to the network input."""

    def get_3rd_point(a, b):
        direct = a - b
        return b + np.array([-direct[1], direct[0]], dtype=np.float32)

    def rotate_point(pt, angle_rad):
        sn, cs = np.sin(angle_rad), np.cos(angle_rad)
        return np.array([pt[0] * cs - pt[1] * sn, pt[0] * sn + pt[1] * cs], dtype=np.float32)

    if not isinstance(scale, (np.ndarray, list)):
        scale = np.array([scale, scale], dtype=np.float32)

    scale_tmp = np.asarray(scale, dtype=np.float32) * 200.0
    src_w = scale_tmp[0]
    dst_w, dst_h = output_size

    rot_rad = np.pi * rot / 180.0
    src_dir = rotate_point(np.array([0, src_w * -0.5], dtype=np.float32), rot_rad)
    dst_dir = np.array([0, dst_w * -0.5], dtype=np.float32)

    center = np.asarray(center, dtype=np.float32)

    src = np.zeros((3, 2), dtype=np.float32)
    dst = np.zeros((3, 2), dtype=np.float32)

    src[0] = center + scale_tmp * shift
    src[1] = center + src_dir + scale_tmp * shift
    src[2] = get_3rd_point(src[0], src[1])

    dst[0] = [dst_w * 0.5, dst_h * 0.5]
    dst[1] = np.array([dst_w * 0.5, dst_h * 0.5], dtype=np.float32) + dst_dir
    dst[2] = get_3rd_point(dst[0], dst[1])

    if inv:
        transform = cv2.getAffineTransform(np.float32(dst), np.float32(src))
    else:
        transform = cv2.getAffineTransform(np.float32(src), np.float32(dst))

    return transform.astype(np.float32)


def affine_2x3_to_3x3(transform):
    """Convert a 2x3 affine matrix to homogeneous 3x3 form."""
    out = np.eye(3, dtype=np.float32)
    out[:2] = np.asarray(transform, dtype=np.float32)
    return out


def preprocess_frame(frame_bgr):
    """Convert one BGR frame to the normalized input tensor and the geometric metadata of MVGFormer."""
    out_w, out_h = map(int, config.NETWORK.IMAGE_SIZE)

    image_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    image_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    h, w = frame_bgr.shape[:2]
    center = np.array([w / 2.0, h / 2.0], dtype=np.float32)

    crop_h = float(h)
    crop_w = crop_h * (out_w / out_h)
    scale = np.array([crop_w / 200.0, crop_h / 200.0], dtype=np.float32)

    transform = get_affine_transform(center, scale, 0, (out_w, out_h), inv=0)
    inverse_transform = get_affine_transform(center, scale, 0, (out_w, out_h), inv=1)

    warped_bgr = cv2.warpAffine(frame_bgr, transform, (out_w, out_h), flags=cv2.INTER_LINEAR)
    warped_rgb = cv2.cvtColor(warped_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    normalized = (warped_rgb - image_mean) / image_std

    tensor = torch.from_numpy(normalized.transpose(2, 0, 1)).float()

    return tensor, center, scale, affine_2x3_to_3x3(transform), affine_2x3_to_3x3(inverse_transform)

### 6.2 View metadata

In addition to the image tensors, MVGFormer expects a metadata dictionary for each camera view with the frame and sequence identifiers, the camera calibration, the image centre and scale, the forward and inverse affine transformations, and a set of joint arrays required by the original data interface (`joints_3d`, `joints_2d`, `joints_3d_voxelpose_pred`, ...).

The joint arrays of the metadata (`joints_3d`, `joints_3d_vis`, `joints_2d`, `joints_2d_vis`) exist only to satisfy the interface of the original code, which reads these keys unconditionally; with the reference-point initialization `sample_space` used here, the ground-truth values are not used for the queries. By default (`USE_GT_IN_META_FOR_COMPAT = False`) the arrays are filled with zeros, so **no ground truth is given to the model**. Setting the option to `True` fills them with the ground-truth pose; the final query is selected from the model outputs alone in both cases (Section 6.3), and the test notebook `MVGFormer_test_colab.ipynb` of the repository runs the real model on the first frames of a sequence both ways with the same weights, images and seed, and checks that the predictions and the selected queries are the same. In the run on a Tesla T4 the maximum difference was 0.0013 mm (tolerance 0.5 mm; two identical runs already differ by 0.0010 mm because of GPU non-determinism), with the same selected queries.

`project_pose_simple` projects a 3D pose to the image plane with the camera parameters to fill the 2D fields; the distortion is not modelled in this projection.

In [ ]:
# Build the metadata required for each camera view

def project_pose_simple(points3d_mm, camera):
    """Project 3D points (mm) onto the image plane of a camera, without lens distortion."""
    X = np.asarray(points3d_mm, dtype=np.float32).reshape(-1, 3)
    R = np.asarray(camera["R"], dtype=np.float32).reshape(3, 3)
    T = np.asarray(camera["T"], dtype=np.float32).reshape(3, 1)

    X_cam = (R @ X.T + T).T
    z = np.where(np.abs(X_cam[:, 2:3]) < 1e-6, 1e-6, X_cam[:, 2:3])

    fx, fy = float(camera["fx"]), float(camera["fy"])
    cx, cy = float(camera["cx"]), float(camera["cy"])

    u = fx * (X_cam[:, 0:1] / z) + cx
    v = fy * (X_cam[:, 1:2] / z) + cy

    return np.concatenate([u, v], axis=1).astype(np.float32)


def build_joints_3d_voxelpose_pred(gt_raw_mm=None):
    """Build the joints_3d_voxelpose_pred array expected by MVGFormer."""
    out = np.zeros((int(config.MULTI_PERSON.MAX_PEOPLE_NUM), int(config.DECODER.num_keypoints), 5), dtype=np.float32)

    if gt_raw_mm is not None:
        out[0, :, :3] = maybe_to_mm(gt_raw_mm)
        out[0, :, 3] = 1.0
        out[0, :, 4] = 1.0

    return out


def make_meta_for_view(subject, exercise, frame_idx, cam_name, cam_id, camera, center, scale,
                       affine_trans, inv_affine_trans, gt_common15_mm=None, use_gt_in_meta=True):
    """Create the metadata dictionary of one camera view."""
    max_people = int(config.MULTI_PERSON.MAX_PEOPLE_NUM)
    num_joints = int(config.DECODER.num_keypoints)

    joints_3d = np.zeros((max_people, num_joints, 3), dtype=np.float32)
    joints_3d_vis = np.zeros((max_people, num_joints, 3), dtype=np.float32)
    joints_2d = np.zeros((max_people, num_joints, 2), dtype=np.float32)
    joints_2d_vis = np.zeros((max_people, num_joints, 2), dtype=np.float32)

    gt_raw_mm = None

    if use_gt_in_meta and gt_common15_mm is not None:
        gt_raw_mm = common15_to_mvg_raw(gt_common15_mm)
        joints_3d[0] = gt_raw_mm
        joints_3d_vis[0] = 1.0
        joints_2d[0] = project_pose_simple(gt_raw_mm, camera)
        joints_2d_vis[0] = 1.0

    name = f"video_direct::{subject}/{exercise}/{cam_name}/{frame_idx:06d}"

    return {
        "image": name,
        "filename": name,
        "imgnum": int(frame_idx),
        "frame": int(frame_idx),
        "num_person": 1,
        "joints_3d": joints_3d,
        "joints_3d_vis": joints_3d_vis,
        "joints": joints_2d,
        "joints_vis": joints_2d_vis,
        "joints_2d": joints_2d,
        "joints_2d_vis": joints_2d_vis,
        "joints_3d_voxelpose_pred": build_joints_3d_voxelpose_pred(gt_raw_mm),
        "center": np.asarray(center, dtype=np.float32),
        "scale": np.asarray(scale, dtype=np.float32),
        "rotation": 0,
        "affine_trans": np.asarray(affine_trans, dtype=np.float32),
        "inv_affine_trans": np.asarray(inv_affine_trans, dtype=np.float32),
        "camera": camera,
        "camera_id": cam_id,
        "camera_name": cam_name,
        "source": "video_direct",
        "exercise": exercise,
    }


def stack_meta_values(values, device):
    """Stack metadata values of a batch and move the numerical data to the target device."""
    value0 = values[0]

    if isinstance(value0, torch.Tensor):
        out = torch.stack([v.float() if v.dtype == torch.float64 else v for v in values], dim=0)
        return out.to(device, non_blocking=True)

    if isinstance(value0, np.ndarray):
        if value0.dtype.kind in ["U", "S", "O"]:
            return list(values)

        array = np.stack(values, axis=0)
        array = array.astype(np.float32) if array.dtype == np.float64 else array
        return torch.from_numpy(array).to(device, non_blocking=True)

    if isinstance(value0, np.generic):
        array = np.asarray([v.item() for v in values])
        array = array.astype(np.float32) if array.dtype == np.float64 else array
        return torch.from_numpy(array).to(device, non_blocking=True)

    if isinstance(value0, (int, float, bool, np.integer, np.floating, np.bool_)):
        array = np.asarray(values)
        array = array.astype(np.float32) if array.dtype == np.float64 else array
        return torch.from_numpy(array).to(device, non_blocking=True)

    if isinstance(value0, str):
        return list(values)

    if isinstance(value0, dict):
        return {key: stack_meta_values([v[key] for v in values], device) for key in value0.keys()}

    return list(values)


def collate_meta_views_batch(meta_by_view, device):
    """Batch the metadata of each camera view independently."""
    return [
        {key: stack_meta_values([meta[key] for meta in view_metas], device) for key in view_metas[0].keys()}
        for view_metas in meta_by_view
    ]

### 6.3 Query selection

For every frame, MVGFormer produces one 3D pose for each of its 1024 queries together with a classification logit per query, from which a confidence score is obtained (softmax probability of the "person" class). The final pose is chosen **from the model outputs only**, without comparing the candidates with the ground truth:

1. Every candidate is converted to millimetres, reordered to COMMON15 and root-aligned on the pelvis.
2. A candidate is *geometrically valid* if
   - at least `MIN_ACTIVE_JOINTS` = 14 of its 15 joints are non-zero,
   - the sum of its 14 bone lengths is between 1000 and 12000 mm, and
   - its longest bone is between 100 and 2500 mm.
3. The valid candidate with the highest score is selected.
4. If no candidate is valid, the one with the largest total bone length is used as a fallback (`fallback_body`). The number of frames in which this happens is stored in the results.

In [ ]:
# Extract the MVGFormer queries and select the final prediction without using the ground truth

MIN_ACTIVE_JOINTS = 14
BODY_SIZE_RANGE_MM = (1000.0, 12000.0)    # sum of the bone lengths
BODY_MAX_RANGE_MM = (100.0, 2500.0)       # longest bone

MVG_RAW_TO_COMMON15_NP = np.asarray(MVG_RAW_TO_COMMON15, dtype=np.int64)
BONES_COMMON15_NP = np.asarray(BONES_COMMON15, dtype=np.int64)


def extract_outputs_coord_batch(out, num_joints=None):
    """Extract the predicted 3D coordinates and the query scores from the output of MVGFormer."""
    if num_joints is None:
        num_joints = int(config.DECODER.num_keypoints)

    pred_logits = out["pred_logits"]
    pred_poses = out["pred_poses"]
    coord = pred_poses["outputs_coord"] if isinstance(pred_poses, dict) else pred_poses
    coord = coord.detach()

    batch_size = int(pred_logits.shape[0])
    pred_raw = coord if coord.ndim == 4 else coord.reshape(batch_size, -1, num_joints, 3)
    scores = pred_logits.softmax(-1)[..., 1]

    return pred_raw.detach().cpu().numpy().astype(np.float32), scores.detach().cpu().numpy().astype(np.float32)


def select_best_query_no_gt_batch(out):
    """Select, for each frame, the highest-scoring geometrically valid query (ground truth is not used)."""
    pred_raw, scores = extract_outputs_coord_batch(out)

    if np.nanmax(np.abs(pred_raw)) < 20:
        pred_raw *= 1000.0

    pred_common = pred_raw[:, :, MVG_RAW_TO_COMMON15_NP, :]
    pred_root = pred_common - pred_common[:, :, 0:1, :]

    bone_lengths = np.linalg.norm(
        pred_root[:, :, BONES_COMMON15_NP[:, 0], :] - pred_root[:, :, BONES_COMMON15_NP[:, 1], :], axis=-1
    )
    body_size = bone_lengths.sum(axis=-1)
    body_max = bone_lengths.max(axis=-1)
    nonzero = (np.linalg.norm(pred_root, axis=-1) > 1e-6).sum(axis=-1)

    valid = (
        (nonzero >= MIN_ACTIVE_JOINTS)
        & (body_size >= BODY_SIZE_RANGE_MM[0]) & (body_size <= BODY_SIZE_RANGE_MM[1])
        & (body_max >= BODY_MAX_RANGE_MM[0]) & (body_max <= BODY_MAX_RANGE_MM[1])
    )

    poses, details = [], []

    for index in range(pred_raw.shape[0]):
        if np.any(valid[index]):
            valid_queries = np.where(valid[index])[0]
            query = int(valid_queries[np.argmax(scores[index, valid_queries])])
            mode = "score_valid_body"
        else:
            query = int(np.argmax(body_size[index]))
            mode = "fallback_body"

        details.append({
            "q": query,
            "score": float(scores[index, query]),
            "body_size": float(body_size[index, query]),
            "body_max": float(body_max[index, query]),
            "nonzero": int(nonzero[index, query]),
            "valid": bool(valid[index, query]),
            "mode": mode,
        })
        poses.append(pred_root[index, query].astype(np.float32))

    return np.asarray(poses, dtype=np.float32), details

### 6.4 Efficient video access and multi-view triangulation

Processing four complete videos directly from Google Drive is dominated by I/O. The pipeline therefore

- **copies the four videos of a sequence to the local disk** before inference (`USE_LOCAL_VIDEO_CACHE`) and removes them afterwards,
- **reads the frames sequentially**: consecutive frames are read directly, forward jumps (`FRAME_STRIDE > 1`) use `grab()`, and random access is used only when needed.

It also replaces the point-by-point triangulation routine of the original code with a **vectorized implementation**. The linear systems of all points of a batch are assembled simultaneously and solved with a batched singular value decomposition; the confidence values are used as weights of the equations when they are available.

In [ ]:
# Local video cache and efficient frame access

def cache_videos(subject, exercise, raw_video_paths):
    """Copy the four videos of a sequence to the local disk (if enabled); return their paths and the copies."""
    if not USE_LOCAL_VIDEO_CACHE:
        return raw_video_paths, []

    local_paths, cached_files = {}, []

    for cam_id, src_path in raw_video_paths.items():
        src_path = Path(src_path)
        local_dir = LOCAL_VIDEO_CACHE_ROOT / subject / exercise / cam_id
        local_dir.mkdir(parents=True, exist_ok=True)
        local_path = local_dir / src_path.name

        try:
            copy_needed = (not local_path.exists()) or local_path.stat().st_size != src_path.stat().st_size
        except OSError:
            copy_needed = True

        if copy_needed:
            shutil.copy2(src_path, local_path)

        local_paths[cam_id] = local_path
        cached_files.append(local_path)

    return local_paths, cached_files


def cleanup_cached_files(cached_files):
    """Delete the local copies of the videos (if enabled)."""
    if not CLEAR_VIDEO_CACHE:
        return

    for path in cached_files:
        try:
            Path(path).unlink(missing_ok=True)
        except OSError:
            pass


def read_frame_smart(cap, target_idx, last_pos_by_cam, cam_id):
    """Read a frame, prioritizing sequential access."""
    target_idx = int(target_idx)
    last_pos = int(last_pos_by_cam.get(cam_id, -1))

    if last_pos >= 0 and target_idx > last_pos + 1:
        # Forward jump: skip the intermediate frames without decoding them
        while last_pos < target_idx - 1:
            if not cap.grab():
                raise RuntimeError(f"Unable to advance to frame {target_idx} in camera {cam_id}")

            last_pos += 1
    elif last_pos != target_idx - 1:
        # First frame, or backward jump
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)

    ok, frame = cap.read()

    if not ok or frame is None:
        raise RuntimeError(f"Unable to read frame {target_idx} from camera {cam_id}")

    last_pos_by_cam[cam_id] = target_idx

    return frame

In [ ]:
# Vectorized multi-view triangulation

def install_fast_triangulation_patch():
    """Replace the point-by-point triangulation of MVGFormer by a batched SVD implementation."""
    import mvn.utils.multiview as multiview_mod

    if not hasattr(multiview_mod, "_orig_triangulate_batch_of_points_batch_version"):
        multiview_mod._orig_triangulate_batch_of_points_batch_version = (
            multiview_mod.triangulate_batch_of_points_batch_version
        )

    def triangulate_batch_of_points_batch_version_fast(proj_matricies_batch, points_batch, confidences_batch=None, solver="linalg"):
        """Triangulate batches of multi-view 2D points using batched SVD."""
        P = proj_matricies_batch
        pts = points_batch
        conf = confidences_batch

        if not torch.is_tensor(P):
            P = torch.as_tensor(P)

        if not torch.is_tensor(pts):
            pts = torch.as_tensor(pts, device=P.device)

        P = P.to(device=pts.device, dtype=torch.float32)
        pts = pts.to(device=P.device, dtype=torch.float32)

        squeeze_batch = False

        if P.ndim == 3:
            P = P.unsqueeze(0)
            squeeze_batch = True

        if pts.ndim == 3:
            pts = pts.unsqueeze(0)

        batch_size = P.shape[0]
        num_views = P.shape[1]

        if pts.shape[1] == num_views and pts.shape[2] != num_views:
            pts = pts.permute(0, 2, 1, 3).contiguous()

        if pts.shape[2] != num_views:
            raise RuntimeError(f"Incompatible triangulation shapes: P={tuple(P.shape)}, pts={tuple(pts.shape)}")

        num_points = pts.shape[1]
        P_exp = P[:, None, :, :, :]

        x = pts[..., 0:1]
        y = pts[..., 1:2]

        row_x = x * P_exp[:, :, :, 2, :] - P_exp[:, :, :, 0, :]
        row_y = y * P_exp[:, :, :, 2, :] - P_exp[:, :, :, 1, :]
        A = torch.stack([row_x, row_y], dim=3)

        if conf is not None:
            if not torch.is_tensor(conf):
                conf = torch.as_tensor(conf, device=P.device)

            conf = conf.to(device=P.device, dtype=torch.float32)

            if conf.ndim == 4 and conf.shape[-1] == 1:
                conf = conf[..., 0]

            if conf.ndim == 2:
                conf = conf.unsqueeze(0)

            if conf.shape[1] == num_views and conf.shape[2] != num_views:
                conf = conf.permute(0, 2, 1).contiguous()

            if conf.shape[1] != num_points or conf.shape[2] != num_views:
                raise RuntimeError(
                    f"Incompatible confidence shape: conf={tuple(conf.shape)}, expected={(batch_size, num_points, num_views)}"
                )

            A = A * conf[:, :, :, None, None]

        A = A.reshape(batch_size, num_points, num_views * 2, 4)
        _, _, Vh = torch.linalg.svd(A, full_matrices=True)

        points_h = Vh[..., -1, :]
        denominator = points_h[..., 3:4]
        denominator = torch.where(torch.abs(denominator) < 1e-8, torch.full_like(denominator, 1e-8), denominator)

        points_3d = points_h[..., :3] / denominator

        return points_3d[0] if squeeze_batch else points_3d

    multiview_mod.triangulate_batch_of_points_batch_version = triangulate_batch_of_points_batch_version_fast

    return multiview_mod.triangulate_batch_of_points_batch_version


triangulation_fn = install_fast_triangulation_patch()
print("Vectorized triangulation installed.")

### 6.5 Inference of a sequence

`process_sequence` runs the complete pipeline for one subject-exercise pair:

1. Locate the four videos, optionally copy them to the local disk and count their frames.
2. Load the ground truth and the four calibrations.
3. Restrict the sequence to the frames available in **all** cameras and in the ground truth.
4. For each batch of frames: read and preprocess the four views, build the metadata, run MVGFormer and select the prediction of every frame (Section 6.3).
5. Prepare the ground truth for the evaluation (arm exchange and pelvis alignment).

`process_sequence_auto_batch` repeats the sequence with the next smaller batch size if the GPU runs out of memory.

In [ ]:
# Multi-view inference of one sequence

def build_batch(subject, exercise, batch_frame_indices, caps, last_pos_by_cam, cameras, gt_common15_all, use_gt_in_meta):
    """Read and preprocess a batch of synchronized frames; return the model inputs."""
    view_tensors = {cam_id: [] for cam_id in CAMERA_IDS}
    meta_by_view = [[] for _ in CAMERA_IDS]

    for frame_idx in batch_frame_indices:
        gt_common15 = gt_common15_all[frame_idx].astype(np.float32)

        for view_idx, cam_id in enumerate(CAMERA_IDS):
            cam_name = CAM_ID_TO_CAM_NAME[cam_id]
            frame_bgr = read_frame_smart(caps[cam_id], frame_idx, last_pos_by_cam, cam_id)
            tensor, center, scale, affine_trans, inv_affine_trans = preprocess_frame(frame_bgr)

            view_tensors[cam_id].append(tensor)
            meta_by_view[view_idx].append(
                make_meta_for_view(
                    subject, exercise, frame_idx, cam_name, cam_id, cameras[cam_name], center, scale,
                    affine_trans, inv_affine_trans, gt_common15_mm=gt_common15, use_gt_in_meta=use_gt_in_meta,
                )
            )

    input_tensors = [
        torch.stack(view_tensors[cam_id], dim=0).to(device, non_blocking=True) for cam_id in CAMERA_IDS
    ]

    return input_tensors, collate_meta_views_batch(meta_by_view, device)


def predict_batch(input_tensors, meta):
    """Run MVGFormer on a batch of frames and select one pose per frame."""
    # The decoder prints "<field> is missing" for every non-tensor metadata field (filename, camera_id, ...) in each
    # forward pass. These messages are harmless and unrelated to the results, so the standard output is discarded here.
    with torch.inference_mode(), contextlib.redirect_stdout(io.StringIO()):
        out = model(views=input_tensors, meta=meta)

    return select_best_query_no_gt_batch(out)


def process_sequence(subject, exercise, batch_size):
    """Run MVGFormer on one sequence; return the predictions, the ground truth and the frame statistics."""
    seed_everything(SEED)
    started = time.time()

    raw_video_paths = {cam_id: find_video_file(subject, exercise, cam_id) for cam_id in CAMERA_IDS}
    video_paths, cached_files = cache_videos(subject, exercise, raw_video_paths)
    caps = {}

    try:
        frame_counts = {cam_id: count_frames(video_paths[cam_id]) for cam_id in CAMERA_IDS}
        gt_common15_all, gt_path = load_ground_truth(subject, exercise)
        gt_eval_all = to_evaluation_ground_truth(gt_common15_all)
        cameras = load_cameras(subject, exercise)

        frame_indices = select_frame_indices(min(min(frame_counts.values()), len(gt_common15_all)))

        if len(frame_indices) == 0:
            raise RuntimeError(f"No valid frames found for {subject}/{exercise}")

        for cam_id in CAMERA_IDS:
            caps[cam_id] = cv2.VideoCapture(str(video_paths[cam_id]))

            if not caps[cam_id].isOpened():
                raise RuntimeError(f"Unable to open video: {video_paths[cam_id]}")

        last_pos_by_cam = {cam_id: -1 for cam_id in CAMERA_IDS}
        predictions, frame_rows = [], []

        for start in tqdm(range(0, len(frame_indices), batch_size), desc=f"{subject}/{exercise}", leave=True, mininterval=1.0):
            batch_frame_indices = frame_indices[start:start + batch_size]
            batch_started = time.time()

            input_tensors, meta = build_batch(
                subject, exercise, batch_frame_indices, caps, last_pos_by_cam, cameras, gt_common15_all,
                USE_GT_IN_META_FOR_COMPAT,
            )
            batch_poses, details = predict_batch(input_tensors, meta)
            frame_time = (time.time() - batch_started) / max(len(batch_frame_indices), 1)

            for frame_idx, pose, best in zip(batch_frame_indices, batch_poses, details):
                predictions.append(pose.astype(np.float32))
                frame_rows.append({
                    "frame_idx": int(frame_idx),
                    "best_q": best["q"],
                    "best_score": best["score"],
                    "body_size": best["body_size"],
                    "body_max": best["body_max"],
                    "nonzero": best["nonzero"],
                    "mode": best["mode"],
                    "time_sec": float(frame_time),
                })
    finally:
        for cap in caps.values():
            cap.release()

        cleanup_cached_files(cached_files)

    predictions = np.asarray(predictions, dtype=np.float32)
    ground_truth = gt_eval_all[frame_indices][:len(predictions)]

    return {
        "predictions": predictions,
        "ground_truth": ground_truth,
        "gt_path": gt_path,
        "frame_rows": pd.DataFrame(frame_rows),
        "elapsed": time.time() - started,
    }


def process_sequence_auto_batch(subject, exercise):
    """Process a sequence, reducing the batch size after a GPU out-of-memory error."""
    last_error = None

    for batch_size in BATCH_CANDIDATES:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()
            result = process_sequence(subject, exercise, batch_size)
            result["batch_size"] = int(batch_size)

            return result

        except RuntimeError as error:
            if "out of memory" not in str(error).lower():
                raise

            last_error = error
            print(f"CUDA OOM | {subject}/{exercise} | batch={batch_size}. Reducing the batch size.")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()

    raise RuntimeError(f"Unable to process {subject}/{exercise}. Last error: {last_error!r}")

## 7. Evaluation Metrics

Prediction and ground truth are compared in COMMON15, in millimetres and relative to the pelvis, frame by frame:

- **MPJPE:** mean Euclidean distance between corresponding joints.
- **PA-MPJPE:** the same error after a rigid similarity (Procrustes) alignment of **each frame** of the prediction to the ground truth, which removes global rotation, translation and scale.
- **Per-joint MPJPE and PA-MPJPE:** the mean over the frames of the error of each of the 15 joints.

A sequence is summarized by the mean over its frames.

In [ ]:
# Evaluation metrics

def compute_similarity_transform(X, Y):
    """Align the pose X to the pose Y with a similarity transformation (Procrustes analysis)."""
    X = fix_pose(X).astype(np.float64)
    Y = fix_pose(Y).astype(np.float64)

    muX, muY = X.mean(axis=0), Y.mean(axis=0)
    X0, Y0 = X - muX, Y - muY

    normX, normY = np.linalg.norm(X0), np.linalg.norm(Y0)

    if normX < 1e-8 or normY < 1e-8:
        return X.astype(np.float32)

    X0 /= normX
    Y0 /= normY

    U, s, Vt = np.linalg.svd(X0.T @ Y0)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    scale = normY / normX * np.sum(s)
    t = muY - scale * (R @ muX)

    return ((scale * (R @ X.T)).T + t).astype(np.float32)


def evaluate_video_metrics(pred_seq_mm, gt_seq_mm):
    """Compute the sequence-level and per-joint MPJPE and PA-MPJPE."""
    pred = np.asarray(pred_seq_mm, dtype=np.float32)
    gt = np.asarray(gt_seq_mm, dtype=np.float32)

    assert pred.shape == gt.shape, (pred.shape, gt.shape)

    error = np.linalg.norm(pred - gt, axis=-1)
    pred_aligned = np.stack([compute_similarity_transform(pred[t], gt[t]) for t in range(pred.shape[0])])
    error_pa = np.linalg.norm(pred_aligned - gt, axis=-1)

    return {
        "MPJPE_mm": float(error.mean()),
        "PA_MPJPE_mm": float(error_pa.mean()),
        "PerJoint_MPJPE_mm": error.mean(axis=0),
        "PerJoint_PA_MPJPE_mm": error_pa.mean(axis=0),
    }

## 8. Running the Experiment

The experiment is run independently for every subject-exercise pair that has the four videos, the four calibration files and the 3D ground truth. For each sequence (Section 6.5) the notebook

1. runs the multi-view inference,
2. computes MPJPE, PA-MPJPE and the per-joint errors (Section 7),
3. saves the prediction and appends the results to the CSV files.

The run is **resumable**: a sequence whose prediction file and CSV entry already exist is skipped. A sequence that fails is reported and the run continues with the next one (unless `STOP_ON_FIRST_ERROR` is enabled); re-running the cell retries the failed sequences. The random seed is reset before every sequence, so a sequence gives the same result regardless of the order of the run.

**Output files** (under `SAVE_ROOT`, which is `Fit3D_MVGFormer` inside `STORAGE_ROOT`; a subset run uses `Fit3D_MVGFormer_subset` so that it never mixes with the complete experiment):

| File | Content |
|---|---|
| `predictions_3d/<subject>/<exercise>.npy` | 3D predictions `(T, 15, 3)`: COMMON15 skeleton, millimetres, pelvis-centred |
| `csv/results_per_video_mvgformer_video_direct.csv` | `subject`, `video`, `pose_source`, `mode`, `frames`, `frame_stride`, `batch_size_frames`, `MPJPE_mm`, `PA_MPJPE_mm`, `time_sec`, paths, and statistics of the selected queries (`best_q_mode`, `best_score_mean`, `body_size_mean`, `body_max_mean`, `fallback_frames`, `frame_time_mean_sec`) |
| `csv/results_per_joint_mvgformer_video_direct.csv` | `subject`, `video`, `pose_source`, `joint`, `MPJPE_mm`, `PA_MPJPE_mm` for each of the 15 joints |

These predictions are the input of the temporal filtering experiments.

In [ ]:
# Discover the sequences and manage the result files

def discover_cases():
    """Find the subject-exercise pairs that have videos, calibrations and ground truth."""
    subjects = sorted(path.name for path in VIDEO_ROOT.iterdir() if path.is_dir())

    if SUBJECTS is not None:
        subjects = [subject for subject in subjects if subject in set(SUBJECTS)]

    cases = []

    for subject in subjects:
        cam0_dir = next(
            (path for path in (VIDEO_ROOT / subject / "videos" / CAMERA_IDS[0], VIDEO_ROOT / subject / CAMERA_IDS[0]) if path.exists()),
            None,
        )

        if cam0_dir is None:
            print(f"SKIP | {subject} | folder of camera {CAMERA_IDS[0]} not found.")
            continue

        exercises = sorted(path.stem for path in cam0_dir.glob("*.mp4"))

        if EXERCISES is not None:
            exercises = [exercise for exercise in exercises if exercise in set(EXERCISES)]

        if EXCLUDE_WARMUPS:
            exercises = [exercise for exercise in exercises if not exercise.lower().startswith("warmup")]

        found = 0

        for exercise in exercises:
            if MAX_SEQUENCES_PER_SUBJECT is not None and found >= MAX_SEQUENCES_PER_SUBJECT:
                break

            try:
                find_gt_json(subject, exercise)

                for cam_id in CAMERA_IDS:
                    find_video_file(subject, exercise, cam_id)
                    find_camera_param_file(subject, exercise, cam_id)

                cases.append((subject, exercise))
                found += 1

            except Exception as error:
                print(f"SKIP | {subject}/{exercise} | {error}")

    return cases


def save_rows(path, rows):
    """Write rows to a CSV file, replacing any earlier rows of the same sequences."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame(rows)

    if path.exists():
        old = pd.read_csv(path)
        replaced = set(zip(new["subject"].astype(str), new["video"].astype(str)))
        keep = [(str(s), str(v)) not in replaced for s, v in zip(old["subject"], old["video"])]
        new = pd.concat([old[keep], new], ignore_index=True)

    new.to_csv(path, index=False)


def load_completed_sequences(csv_path):
    """Return the (subject, video) pairs already registered in the per-video CSV with the CURRENT configuration.

    Rows produced with another configuration (a different RUN_TAG, e.g. another value of
    USE_GT_IN_META_FOR_COMPAT or an older version of the notebook) are not reused: those sequences are processed again
    and their rows are replaced."""
    if not Path(csv_path).exists():
        return set()

    try:
        table = pd.read_csv(csv_path)

        if "mode" not in table.columns:
            return set()

        table = table[table["mode"].astype(str) == RUN_TAG]
        return set(zip(table["subject"].astype(str), table["video"].astype(str)))
    except Exception:
        return set()


CASES = discover_cases()

print(f"Sequences to process: {len(CASES)} from {len({subject for subject, _ in CASES})} subjects")

for case in CASES[:10]:
    print(" -", "/".join(case))

if len(CASES) > 10:
    print(" ...")

print(f"Predictions: {PRED_DIR}")

In [ ]:
# Run the MVGFormer inference and evaluation

def evaluate_and_save(subject, exercise, result):
    """Evaluate a processed sequence and save its prediction and results; return the per-video row."""
    predictions, ground_truth, frame_rows = result["predictions"], result["ground_truth"], result["frame_rows"]

    if len(predictions) == 0:
        raise RuntimeError(f"No predictions were generated for {subject}/{exercise}")

    metrics = evaluate_video_metrics(predictions, ground_truth)

    pred_path = PRED_DIR / subject / f"{exercise}.npy"
    pred_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(pred_path, predictions.astype(np.float32))

    row = {
        "subject": subject,
        "video": exercise,
        "pose_source": POSE_SOURCE_NAME,
        "mode": RUN_TAG,
        "frames": int(len(predictions)),
        "frame_stride": int(FRAME_STRIDE),
        "batch_size_frames": int(result["batch_size"]),
        "MPJPE_mm": metrics["MPJPE_mm"],
        "PA_MPJPE_mm": metrics["PA_MPJPE_mm"],
        "time_sec": float(result["elapsed"]),
        "pred_path": str(pred_path),
        "gt_path": str(result["gt_path"]),
        "best_q_mode": int(frame_rows["best_q"].mode().iloc[0]),
        "best_score_mean": float(frame_rows["best_score"].mean()),
        "body_size_mean": float(frame_rows["body_size"].mean()),
        "body_max_mean": float(frame_rows["body_max"].mean()),
        "fallback_frames": int((frame_rows["mode"] == "fallback_body").sum()),
        "frame_time_mean_sec": float(frame_rows["time_sec"].mean()),
    }

    save_rows(PER_VIDEO_CSV, [row])
    save_rows(PER_JOINT_CSV, [
        {
            "subject": subject,
            "video": exercise,
            "pose_source": POSE_SOURCE_NAME,
            "joint": joint_name,
            "MPJPE_mm": float(metrics["PerJoint_MPJPE_mm"][index]),
            "PA_MPJPE_mm": float(metrics["PerJoint_PA_MPJPE_mm"][index]),
        }
        for index, joint_name in enumerate(COMMON15_NAMES)
    ])

    print(
        f"{subject}/{exercise} | frames={row['frames']} | batch={row['batch_size_frames']} | "
        f"MPJPE={row['MPJPE_mm']:.2f} mm | PA-MPJPE={row['PA_MPJPE_mm']:.2f} mm | "
        f"fallback frames={row['fallback_frames']} | {row['time_sec']:.1f} s"
    )

    return row


completed = load_completed_sequences(PER_VIDEO_CSV)

if Path(PER_VIDEO_CSV).exists():
    registered = len(pd.read_csv(PER_VIDEO_CSV))

    if registered > len(completed):
        print(f"Note: {registered - len(completed)} sequences registered in the CSV were produced with another configuration "
              f"and will be processed again (current configuration: {RUN_TAG}).")
run_rows, failed = [], []
run_started = time.time()

for subject, exercise in CASES:
    if (subject, exercise) in completed and (PRED_DIR / subject / f"{exercise}.npy").exists():
        print(f"SKIP | {subject}/{exercise} | already processed")
        continue

    try:
        result = process_sequence_auto_batch(subject, exercise)
        run_rows.append(evaluate_and_save(subject, exercise, result))

    except Exception as error:
        failed.append((subject, exercise, repr(error)))
        print(f"ERROR | {subject}/{exercise} | {error!r}")
        traceback.print_exc(limit=8)

        if STOP_ON_FIRST_ERROR:
            break

    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

run_elapsed = time.time() - run_started

print(f"\nFinished: {len(run_rows)} sequences processed in this run, {len(failed)} with errors, {run_elapsed:.1f} s.")

if failed:
    print("Sequences with errors (re-run this cell to retry them):")

    for subject, exercise, message in failed:
        print(f"  {subject}/{exercise}: {message}")

## 9. Output Verification

The next cell checks the integrity of the generated files:

- **Predictions:** every expected file exists, can be loaded, has shape `(T, 15, 3)`, contains no NaN/Inf values and is pelvis-centred; there are no unexpected files.
- **Frame counts:** the number of frames equals the number of frames selected for the sequence, that is, those available in the four cameras and in the ground truth, sampled with `FRAME_STRIDE` and limited by `MAX_FRAMES_PER_VIDEO` (`CHECK_FRAME_COUNTS`; it re-opens the videos, so it is slower).
- **Per-video CSV:** required columns, no duplicates, one row per sequence, finite values and a frame count consistent with the prediction file.
- **Per-joint CSV:** required columns, no duplicates, 15 joints per sequence and finite values.

The verdict is `OK` when no problem is found.

In [ ]:
# Verify the generated outputs

CHECK_FRAME_COUNTS = True   # re-opens the videos and the ground truth (slower)


def expected_num_frames(subject, exercise):
    """Number of frames that must be evaluated in a sequence."""
    counts = [count_frames(find_video_file(subject, exercise, cam_id)) for cam_id in CAMERA_IDS]
    gt_frames = len(load_ground_truth(subject, exercise)[0])

    return len(select_frame_indices(min(min(counts), gt_frames)))


def verify_outputs(cases, check_frame_counts=True):
    """Check the generated files; return the list of problems and the number of valid predictions."""
    problems = []

    def report(subject, video, message):
        problems.append({"subject": subject, "video": video, "problem": message})

    expected = set(cases)
    prediction_frames = {}
    valid_files = 0

    # 1. Prediction files
    for subject, video in sorted(expected):
        path = PRED_DIR / subject / f"{video}.npy"

        if not path.exists():
            report(subject, video, "Missing prediction file")
            continue

        try:
            prediction = np.load(path)
        except Exception as error:
            report(subject, video, f"Prediction cannot be loaded: {error}")
            continue

        if prediction.ndim != 3 or prediction.shape[1:] != (15, 3):
            report(subject, video, f"Unexpected shape {prediction.shape}")
            continue

        if not np.isfinite(prediction).all():
            report(subject, video, "NaN or Inf values in prediction")
            continue

        if not np.allclose(prediction[:, 0], 0.0, atol=1e-3):
            report(subject, video, "Prediction is not pelvis-centred")
            continue

        prediction_frames[(subject, video)] = len(prediction)

        if check_frame_counts:
            expected_frames = expected_num_frames(subject, video)

            if len(prediction) != expected_frames:
                report(subject, video, f"Frames: prediction={len(prediction)}, expected={expected_frames}")
                continue

        valid_files += 1

    if PRED_DIR.is_dir():
        for subject_dir in sorted(path for path in PRED_DIR.iterdir() if path.is_dir()):
            for file in sorted(subject_dir.glob("*.npy")):
                if (subject_dir.name, file.stem) not in expected:
                    report(subject_dir.name, file.stem, "Unexpected prediction file")

    # 2. Per-video CSV
    if not PER_VIDEO_CSV.exists():
        report("", "", "Per-video CSV not found")
    else:
        table = pd.read_csv(PER_VIDEO_CSV)
        required = {"subject", "video", "pose_source", "mode", "frames", "MPJPE_mm", "PA_MPJPE_mm", "time_sec"}

        if required - set(table.columns):
            report("", "", f"Per-video CSV lacks columns {sorted(required - set(table.columns))}")
        else:
            if table.duplicated(subset=["subject", "video"]).any():
                report("", "", "Duplicated rows in per-video CSV")

            rows = set(zip(table["subject"].astype(str), table["video"].astype(str)))

            for subject, video in sorted(expected - rows):
                report(subject, video, "Missing row in per-video CSV")

            for subject, video in sorted(rows - expected):
                report(subject, video, "Unexpected row in per-video CSV")

            for column in ["MPJPE_mm", "PA_MPJPE_mm", "time_sec"]:
                invalid = int((~np.isfinite(pd.to_numeric(table[column], errors="coerce"))).sum())

                if invalid:
                    report("", "", f"{invalid} invalid values in {column}")

            for _, row in table.iterrows():
                pair = (str(row["subject"]), str(row["video"]))

                if pair in prediction_frames and int(row["frames"]) != prediction_frames[pair]:
                    report(pair[0], pair[1], f"CSV frames={int(row['frames'])}, prediction frames={prediction_frames[pair]}")

                if pair in expected and str(row["mode"]) != RUN_TAG:
                    report(pair[0], pair[1], f"Incorrect run configuration: {row['mode']} (expected {RUN_TAG})")

    # 3. Per-joint CSV
    if not PER_JOINT_CSV.exists():
        report("", "", "Per-joint CSV not found")
    else:
        table = pd.read_csv(PER_JOINT_CSV)
        required = {"subject", "video", "pose_source", "joint", "MPJPE_mm", "PA_MPJPE_mm"}

        if required - set(table.columns):
            report("", "", f"Per-joint CSV lacks columns {sorted(required - set(table.columns))}")
        else:
            if table.duplicated(subset=["subject", "video", "joint"]).any():
                report("", "", "Duplicated rows in per-joint CSV")

            joints_per_video = table.groupby(["subject", "video"])["joint"].nunique()

            for (subject, video), count in joints_per_video.items():
                if count != len(COMMON15_NAMES):
                    report(subject, video, f"{count} joints in per-joint CSV (expected {len(COMMON15_NAMES)})")

            for subject, video in sorted(expected - set(joints_per_video.index)):
                report(subject, video, "Missing sequence in per-joint CSV")

            for column in ["MPJPE_mm", "PA_MPJPE_mm"]:
                invalid = int((~np.isfinite(pd.to_numeric(table[column], errors="coerce"))).sum())

                if invalid:
                    report("", "", f"{invalid} invalid per-joint values in {column}")

    return problems, valid_files


problems, valid_files = verify_outputs(CASES, CHECK_FRAME_COUNTS)

# The complete experiment must contain 8 subjects x 28 exercises (224 sequences)
if not IS_SUBSET:
    cases_per_subject = pd.Series([case[0] for case in CASES]).value_counts()

    if len(CASES) != 224 or len(cases_per_subject) != 8 or not (cases_per_subject == 28).all():
        problems.append({
            "subject": "", "video": "",
            "problem": f"Incomplete experiment: {len(CASES)} sequences found (expected 224 = 8 subjects x 28); "
                       f"per subject: {cases_per_subject.to_dict()}",
        })

print(f"Expected sequences: {len(CASES)}")
print(f"Valid prediction files: {valid_files}/{len(CASES)}")

if problems:
    print(f"\nProblems detected: {len(problems)}")
    print(pd.DataFrame(problems).head(50).to_string(index=False))
else:
    print("\nNo structural problems detected.")

print("\nVERDICT: " + ("OK" if not problems else "REVIEW THE PROBLEMS LISTED ABOVE"))

## 10. Summary

The last cell summarizes the run. For the complete experiment it also prints the PA-MPJPE (COMMON15) reported in the paper, per subject and as the mean of the subject means. Small differences with respect to the paper may appear with a different GPU or library versions.

In [ ]:
# Summary of the run

PAPER_PA_MPJPE_MM = {   # PA-MPJPE (mm), COMMON15, MVGFormer, complete experiment (per subject and mean of the subjects)
    "s03": 61.672, "s04": 60.299, "s05": 67.718, "s07": 70.701,
    "s08": 77.136, "s09": 71.240, "s10": 55.807, "s11": 68.952,
}
PAPER_MEAN_MM = 66.691

if not PER_VIDEO_CSV.exists():
    print("No results found: check the messages printed in Section 8.")
else:
    summary = pd.read_csv(PER_VIDEO_CSV)
    by_subject = summary.groupby("subject")["PA_MPJPE_mm"].mean()

    print(f"Model: MVGFormer q1024 | skeleton: COMMON15 | configuration: {RUN_TAG}")
    print(f"Sequences evaluated: {len(summary)} | frames: {int(summary['frames'].sum())}")
    print(f"MPJPE (mean over sequences):    {summary['MPJPE_mm'].mean():.2f} mm")
    print(f"PA-MPJPE (mean over sequences): {summary['PA_MPJPE_mm'].mean():.2f} mm")
    print(f"PA-MPJPE (frame-weighted):      {np.average(summary['PA_MPJPE_mm'], weights=summary['frames']):.2f} mm")

    if "fallback_frames" in summary.columns:
        fallback = int(summary["fallback_frames"].sum())
        print(f"Frames without a geometrically valid query (fallback): {fallback} ({100.0 * fallback / max(int(summary['frames'].sum()), 1):.2f} %)")

    if IS_SUBSET:
        print("Subset run: the values are not comparable with the complete experiment of the paper.")
    else:
        comparison = pd.DataFrame({"obtained": by_subject, "paper": pd.Series(PAPER_PA_MPJPE_MM)}).round(3)
        comparison.loc["mean of subjects"] = [by_subject.mean(), PAPER_MEAN_MM]
        print("\nPA-MPJPE per subject (mm):")
        print(comparison.round(3).to_string())
        print(f"\n(224 sequences expected; evaluated: {len(summary)})")

## Troubleshooting

| Symptom | Solution |
|---|---|
| `Deformable*.so was not generated` | The build log is printed (last 80 lines). Use a GPU runtime with the CUDA toolkit, restart the runtime and run Section 2 again. |
| `MVGFormer checkpoint not found` | Place `mvgformer_q1024_model.pth.tar` in `STORAGE_ROOT/checkpoints/`. |
| `Skipped parameters (shape mismatch)` is not 0 | The checkpoint does not correspond to the `q1024` configuration; use the `q1024` weights released by the authors. |
| `CUDA out of memory` | Handled automatically: the sequence is repeated with the next value of `BATCH_CANDIDATES`. If it also fails with `1`, free the GPU memory (restart the runtime). |
| `SKIP` messages (video, calibration or ground truth not found) | Check the folder layout of Section 1 for that subject and exercise. |
| Reading the videos is very slow | Keep `USE_LOCAL_VIDEO_CACHE = True`; reading four videos directly from Google Drive is much slower. |
| The Colab disk is full | Keep `CLEAR_VIDEO_CACHE = True`, so that the local copies of the videos are deleted after every sequence. |
| Results of a previous run are reused | Delete the CSV files and the prediction folder under `SAVE_ROOT` to recompute everything. |